In [1]:
pip install cohere python-dotenv networkx

Note: you may need to restart the kernel to use updated packages.


In [2]:
import cohere
import json
import networkx as nx
from dotenv import load_dotenv
import os
import time

load_dotenv()

api_key = os.getenv("COHERE_API_KEY")

if not api_key:
    raise ValueError("COHERE_API_KEY not found in environment. Check your .env file.")
print("API key loaded.")

API key loaded.


In [4]:
co = cohere.Client(api_key)

In [5]:
def chat_with_retry(max_retries=4, **kwargs):
    """Wraps co.chat with exponential backoff on TooManyRequestsError."""
    for attempt in range(max_retries):
        try:
            return co.chat(**kwargs)
        except Exception as e:
            if "TooManyRequests" in type(e).__name__ or "429" in str(e):
                wait = 15 * (2 ** attempt)  # 15s, 30s, 60s, 120s
                print(f"  Rate limited — waiting {wait}s (attempt {attempt + 1}/{max_retries})...")
                time.sleep(wait)
            else:
                raise  # re-raise non-rate-limit errors immediately
    raise RuntimeError("Max retries exceeded on Cohere rate limit.")

In [6]:
# ── Dataset Configuration ──────────────────────────────────────────────────
DATASET_NAME   = "sci"
DATA_PATH      = "../datasets/sci_wiki_unprocessed.jsonl"
CACHE_PATH     = "../datasets/graph_cache_sci.json"
CONTENT_FIELD  = "text"
TITLE_FIELD    = "title"

print(f"Dataset : {DATASET_NAME}")
print(f"Data    → {DATA_PATH}")
print(f"Cache   → {CACHE_PATH}")

Dataset : sci
Data    → ../datasets/sci_wiki_unprocessed.jsonl
Cache   → data/graph_cache_sci.json


In [7]:
def extract_entities_llm(text):
    prompt = f"""Extract key entities from the text.

Return ONLY a valid JSON array of entity name strings. Example: ["Google", "London", "Alan Turing"]

Focus on: People, Organizations, Places, Technologies, Events.

TEXT:
{text[:3000]}
"""

    response = chat_with_retry(
        model="command-r7b-12-2024",
        message=prompt,
        temperature=0.2
    )

    raw = response.text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()
    try:
        entities = json.loads(raw)
        return [e for e in entities if isinstance(e, str)]
    except json.JSONDecodeError:
        return []

In [8]:
def extract_triplets(text):
    prompt = f"""Extract knowledge graph triplets from the text.

Return ONLY a valid JSON array of arrays, where each inner array is exactly 3 strings:
[["subject", "relation", "object"], ...]

Rules:
- Use short, clean entity names (2-4 words max)
- Use simple verb phrases as relations (e.g. "founded", "is part of", "developed")
- Return 5-15 triplets
- Do NOT include any explanation, just the JSON array

Text:
{text[:3000]}
"""

    response = chat_with_retry(
        model="command-r7b-12-2024",
        message=prompt,
        temperature=0
    )

    raw = response.text.strip()
    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()
    try:
        triplets = json.loads(raw)
        # Validate: keep only well-formed [subject, relation, object] entries
        return [t for t in triplets if isinstance(t, list) and len(t) == 3]
    except json.JSONDecodeError:
        return []

In [9]:
# sci_wiki_unprocessed.jsonl — one JSON object per line
docs = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            docs.append(json.loads(line))

print(f"Loaded {len(docs)} docs  |  sample title: {docs[0][TITLE_FIELD]}")

def save_graph(graph, path):
    data = nx.node_link_data(graph)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)
    print(f"Graph saved to {path}")

def load_graph(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    graph = nx.node_link_graph(data)
    print(f"Graph loaded from cache ({path})")
    return graph

def build_graph(documents, content_field, title_field, limit=None):
    graph = nx.Graph()
    subset = documents[:limit] if limit else documents
    for i, doc in enumerate(subset):
        print(f"Processing doc {i+1}/{len(subset)}: {doc[title_field]}")
        triplets = extract_triplets(doc[content_field])
        for t in triplets:
            try:
                subject, relation, obj = t[0], t[1], t[2]
                graph.add_edge(subject.strip(), obj.strip(), relation=relation.strip())
            except (IndexError, TypeError):
                continue
        time.sleep(0.5)
    return graph

if os.path.exists(CACHE_PATH):
    G = load_graph(CACHE_PATH)
else:
    G = build_graph(docs, CONTENT_FIELD, TITLE_FIELD, limit=100)
    save_graph(G, CACHE_PATH)

print(f"\nNodes: {len(G.nodes)}")
print(f"Edges: {len(G.edges)}")

Loaded 20 docs  |  sample title: Penicillin
Processing doc 1/20: Penicillin
Processing doc 2/20: CRISPR
Processing doc 3/20: General relativity
Processing doc 4/20: Quantum mechanics
Processing doc 5/20: Transistor
Processing doc 6/20: DNA
Processing doc 7/20: Vaccine
Processing doc 8/20: Internet
Processing doc 9/20: Black hole
Processing doc 10/20: Artificial intelligence
Processing doc 11/20: Periodic table
Processing doc 12/20: Electricity
Processing doc 13/20: Steam engine
Processing doc 14/20: Telescope
Processing doc 15/20: Evolution
Processing doc 16/20: Germ theory of disease
Processing doc 17/20: Antimicrobial resistance
Processing doc 18/20: Semiconductor
Processing doc 19/20: Nuclear fission
Processing doc 20/20: Photosynthesis
Graph saved to data/graph_cache_sci.json

Nodes: 454
Edges: 366


In [10]:
def traverse(graph, seeds, depth=2):
    visited = set()
    result = set()

    frontier = [(s, 0) for s in seeds]

    while frontier:
        node, d = frontier.pop()

        if node in visited or d > depth:
            continue

        visited.add(node)
        result.add(node)

        for neighbor in graph.neighbors(node):
            frontier.append((neighbor, d + 1))

    return result

In [11]:
def build_context(graph, nodes):
    context = []

    for n in nodes:
        for neighbor in graph.neighbors(n):
            relation = graph[n][neighbor].get("relation", "related_to")
            context.append(f"{n} -[{relation}]-> {neighbor}")

    return "\n".join(context)

In [12]:
def answer(query, context):
    prompt = f"""You are a GraphRAG assistant.

Use ONLY the knowledge graph below to answer the question.

GRAPH:
{context}

QUESTION:
{query}

Provide a clear, factual answer using the relationships shown above.
"""

    response = chat_with_retry(
        model="command-r7b-12-2024",
        message=prompt,
        temperature=0.3
    )

    return response.text

In [13]:
def get_seed_nodes(query, graph):
    """Find graph nodes that are relevant to the query.
    
    First tries direct substring matching against node names.
    Falls back to LLM entity extraction if no matches found.
    """
    query_lower = query.lower()
    seeds = []

    # Direct match: node name appears in query or query contains node name
    for node in graph.nodes:
        node_lower = str(node).lower()
        if node_lower in query_lower or query_lower in node_lower:
            seeds.append(node)

    # Fallback: use LLM to extract entities from query and match against graph
    if not seeds:
        entities = extract_entities_llm(query)
        for entity in entities:
            entity_lower = entity.lower()
            for node in graph.nodes:
                node_lower = str(node).lower()
                if entity_lower in node_lower or node_lower in entity_lower:
                    seeds.append(node)

    return list(set(seeds))[:5]  # deduplicate and cap at 5 seeds

In [14]:
def graphrag(query, graph=None, return_context=False):
    """Full GraphRAG pipeline: seed → traverse → context → answer.
    
    Args:
        query:          the user question
        graph:          nx.Graph to use (defaults to global G)
        return_context: if True, returns dict {"answer": ..., "context": ...}
                        instead of just the answer string
    """
    g = graph if graph is not None else G

    seeds = get_seed_nodes(query, g)
    if not seeds:
        empty = "No relevant nodes found in the graph for this query."
        return {"answer": empty, "context": ""} if return_context else empty

    subgraph_nodes = traverse(g, seeds)
    context = build_context(g, subgraph_nodes)

    if not context:
        empty = "Found seed nodes but no surrounding relationships to build context from."
        return {"answer": empty, "context": ""} if return_context else empty

    ans = answer(query, context)
    return {"answer": ans, "context": context} if return_context else ans

In [18]:
# ── Step 1: Generate test questions ────────────────────────────────────────
# Generates questions from a sample of the corpus and caches them so the
# same question set is reused across every RAG method your group tests.

QUESTIONS_CACHE = f"../test_questions_{DATASET_NAME}.json"
NUM_DOCS_TO_SAMPLE = 8   # sample more since off-topic docs return empty arrays
QUESTIONS_PER_DOC  = 3   # questions per doc  →  up to 8 × 3 = 24, but ~15 after filtering

def generate_questions_from_doc(doc_text, doc_title):
    prompt = f"""You are creating an evaluation dataset for a GraphRAG system applied to science topics.

Write {QUESTIONS_PER_DOC} complex, multi-part questions that require deep understanding of the text.

Questions should:
- Require synthesising multiple concepts or mechanisms to answer
- Test understanding of causes, effects, processes, or comparisons
- NOT be answerable with a single word, name, or date
- Vary in the concepts they test

Bad examples (too simple, avoid these):
  - "When was DNA discovered?"
  - "What is the speed of light?"

Good examples (complex):
  - "How does the process of natural selection lead to speciation over time?"
  - "Why does quantum tunnelling allow particles to pass through energy barriers that classical physics would forbid?"
  - "How do feedback loops in the carbon cycle amplify or dampen the effects of climate change?"

If the text has no meaningful science content, return: []

Return ONLY a valid JSON array of objects:
[{{"question": "...", "reference_answer": "..."}}]

Title: {doc_title}
Text: {doc_text[:2000]}
"""
    response = chat_with_retry(model="command-r7b-12-2024", message=prompt, temperature=0.3)
    raw = response.text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()
    try:
        items = json.loads(raw)
        return [q for q in items if "question" in q and "reference_answer" in q]
    except json.JSONDecodeError:
        return []

if os.path.exists(QUESTIONS_CACHE):
    with open(QUESTIONS_CACHE, "r", encoding="utf-8") as f:
        test_questions = json.load(f)
    print(f"Loaded {len(test_questions)} cached test questions from {QUESTIONS_CACHE}")
else:
    import random

    sample_docs = random.sample(docs, min(NUM_DOCS_TO_SAMPLE, len(docs)))
    test_questions = []

    for doc in sample_docs:
        print(f"Generating questions for: {doc[TITLE_FIELD]}")
        qs = generate_questions_from_doc(doc[CONTENT_FIELD], doc[TITLE_FIELD])
        test_questions.extend(qs)
        time.sleep(1)

    with open(QUESTIONS_CACHE, "w", encoding="utf-8") as f:
        json.dump(test_questions, f, indent=2, ensure_ascii=False)
    print(f"\nSaved {len(test_questions)} test questions to {QUESTIONS_CACHE}")

for i, q in enumerate(test_questions[:3], 1):
    print(f"\nQ{i}: {q['question']}")

Generating questions for: CRISPR
Generating questions for: Steam engine
Generating questions for: Transistor
Generating questions for: Germ theory of disease
Generating questions for: Periodic table
Generating questions for: Quantum mechanics
Generating questions for: Internet
Generating questions for: Antimicrobial resistance

Saved 24 test questions to ../test_questions_sci.json

Q1: How does the CRISPR-Cas9 system function as an adaptive immune response in prokaryotes, and what are the implications for gene editing?

Q2: What is the historical context of the discovery of CRISPR, and how has it influenced the development of gene editing technologies?

Q3: What are the key characteristics of CRISPR sequences, and how do they contribute to the versatility of the CRISPR-Cas9 technology?


In [16]:
# ── Step 2: LLM Judge ──────────────────────────────────────────────────────
# Scores a single (question, context, answer) triple on 3 metrics.

def llm_judge(question, context, answer_text):
    """Score an answer using an LLM judge.

    Returns a dict with scores (1–5) for:
        faithfulness      – every claim is supported by the context
        answer_relevance  – answer directly addresses the question
        context_relevance – retrieved context was useful for the question

    A reasoning string is also returned for transparency.
    """
    prompt = f"""You are an expert evaluator for Retrieval-Augmented Generation (RAG) systems.

Score the answer below on three criteria using a 1–5 integer scale:

  1 = very poor  |  3 = acceptable  |  5 = excellent

Criteria:
- faithfulness:      Every claim in the answer is directly supported by the provided context.
                     Penalise heavily for hallucination or any statement not grounded in the context.
- answer_relevance:  The answer directly and completely addresses the question with a real answer.
                     IMPORTANT: If the answer says it "cannot answer", "no information", "not found in
                     the graph", or anything similar — that is a FAILED answer. Score it 1.
- context_relevance: The retrieved context actually contained information useful for answering
                     the question. If the context is off-topic or the answer says no info was found,
                     score this 1.

QUESTION:
{question}

RETRIEVED CONTEXT:
{context if context else "(no context retrieved)"}

ANSWER:
{answer_text}

Return ONLY a valid JSON object (no extra text):
{{
  "faithfulness": <1-5>,
  "answer_relevance": <1-5>,
  "context_relevance": <1-5>,
  "reasoning": "<one sentence explaining the scores>"
}}
"""
    response = chat_with_retry(model="command-r7b-12-2024", message=prompt, temperature=0)
    raw = response.text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()
    try:
        scores = json.loads(raw)
        # Clamp scores to valid range
        for key in ("faithfulness", "answer_relevance", "context_relevance"):
            scores[key] = max(1, min(5, int(scores.get(key, 1))))
        return scores
    except (json.JSONDecodeError, ValueError):
        return {"faithfulness": 0, "answer_relevance": 0, "context_relevance": 0,
                "reasoning": "parse error"}

# Quick smoke test
_test = llm_judge(
    "What is machine learning?",
    "Machine learning -[is part of]-> Artificial intelligence",
    "Machine learning is a subfield of artificial intelligence."
)
print("Judge smoke test:", _test)

Judge smoke test: {'faithfulness': 5, 'answer_relevance': 5, 'context_relevance': 5, 'reasoning': 'The answer is accurate, directly addressing the question and supported by the context.'}


In [19]:
# ── Step 3: Run evaluation ─────────────────────────────────────────────────
# Saves results after every question so a crash won't lose progress.
# Re-running this cell will skip already-completed questions automatically.

EVAL_RESULTS_PATH = f"../datasets/eval_results_{DATASET_NAME}.json"

# Resume from existing results if available
if os.path.exists(EVAL_RESULTS_PATH):
    with open(EVAL_RESULTS_PATH, "r", encoding="utf-8") as f:
        eval_results = json.load(f)
    done_questions = {r["question"] for r in eval_results}
    print(f"Resuming — {len(eval_results)} already done, {len(test_questions) - len(done_questions)} remaining.")
else:
    eval_results = []
    done_questions = set()

for i, item in enumerate(test_questions):
    question = item["question"]

    if question in done_questions:
        print(f"[{i+1}/{len(test_questions)}] Skipping (already done): {question[:60]}")
        continue

    print(f"[{i+1}/{len(test_questions)}] {question}")

    for attempt in range(3):
        try:
            result = graphrag(question, return_context=True)
            ans    = result["answer"]
            ctx    = result["context"]
            scores = llm_judge(question, ctx, ans)
            break
        except Exception as e:
            if "TooManyRequests" in type(e).__name__ or "429" in str(e):
                wait = 20 * (attempt + 1)  # 20s, 40s, 60s
                print(f"  Rate limited — waiting {wait}s (attempt {attempt + 1}/3)...")
                time.sleep(wait)
            else:
                raise
    else:
        print(f"  Skipping after 3 failed attempts.")
        continue

    eval_results.append({
        "question":          question,
        "reference_answer":  item.get("reference_answer") or item.get("reference", ""),
        "graphrag_answer":   ans,
        "context":           ctx,
        "faithfulness":      scores["faithfulness"],
        "answer_relevance":  scores["answer_relevance"],
        "context_relevance": scores["context_relevance"],
        "reasoning":         scores.get("reasoning", ""),
    })

    # Save after every question so progress is never lost on a crash
    with open(EVAL_RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(eval_results, f, indent=2, ensure_ascii=False)

    time.sleep(4)  # 2 calls/question × 3s minimum per call; 4s gives safe headroom

print(f"\nDone. {len(eval_results)}/{len(test_questions)} results saved to {EVAL_RESULTS_PATH}")

# ── Summary table ───────────────────────────────────────────────────────────
metrics = ("faithfulness", "answer_relevance", "context_relevance")
averages = {m: sum(r[m] for r in eval_results) / len(eval_results) for m in metrics}

print(f"\n{'='*45}")
print(f"  GraphRAG Evaluation  —  dataset: {DATASET_NAME}")
print(f"{'='*45}")
print(f"  Questions evaluated : {len(eval_results)}")
print(f"  Faithfulness        : {averages['faithfulness']:.2f} / 5")
print(f"  Answer Relevance    : {averages['answer_relevance']:.2f} / 5")
print(f"  Context Relevance   : {averages['context_relevance']:.2f} / 5")
avg_overall = sum(averages.values()) / len(averages)
print(f"  Overall             : {avg_overall:.2f} / 5")
print(f"{'='*45}")

# Per-question breakdown
print("\nPer-question scores (F=Faithfulness, A=Answer Rel., C=Context Rel.):")
print(f"{'#':<4} {'F':>3} {'A':>3} {'C':>3}  Question")
print("-" * 65)
for i, r in enumerate(eval_results, 1):
    q_preview = r["question"][:48] + "…" if len(r["question"]) > 48 else r["question"]
    print(f"{i:<4} {r['faithfulness']:>3} {r['answer_relevance']:>3} {r['context_relevance']:>3}  {q_preview}")

[1/24] How does the CRISPR-Cas9 system function as an adaptive immune response in prokaryotes, and what are the implications for gene editing?
[2/24] What is the historical context of the discovery of CRISPR, and how has it influenced the development of gene editing technologies?
[3/24] What are the key characteristics of CRISPR sequences, and how do they contribute to the versatility of the CRISPR-Cas9 technology?
[4/24] How does the Rankine cycle, as an ideal thermodynamic cycle, contribute to the efficiency of steam engines, and what specific innovation by James Watt further enhanced their performance?
[5/24] What was the significance of Thomas Newcomen's steam engine invention in the 18th century, and how did it contribute to the transition from sail-powered ships to steam-powered vessels?
[6/24] Describe the evolution of steam engines from ancient devices like Hero's aeolipile to the reciprocating piston engines of the Industrial Revolution. What key advancements allowed steam eng